# EGT and Computing payoff matrix

In [ ]:
clinical_data = pd.read_csv("clinical_data.csv", sep=";",header=0, index_col=0)

In [ ]:
clinical_data = clinical_data[['OS_STATUS']]

In [ ]:
reduced_shape_df = pd.DataFrame(data=reduced_shape, index=reduced_data_df.index, columns=[reduced_data_df.columns[i] for i in range(100)])

In [ ]:
reduced_shape_df1 = pd.DataFrame(data=reduced_shape1, index=reduced_data_df1.index, columns=[reduced_data_df1.columns[i] for i in range(100)])

In [ ]:
reduced_shape_df2 = pd.DataFrame(data=reduced_shape2, index=reduced_data_df2.index, columns=[reduced_data_df2.columns[i] for i in range(50)])

In [ ]:
reduced_shape_df3 = pd.DataFrame(data=reduced_shape3, index=reduced_data_df3.index, columns=[reduced_data_df3.columns[i] for i in range(100)])

In [ ]:
def standardize_sample_id(sample_id):
    """Extracts the first four segments of a TCGA sample ID for consistency."""
    return "-".join(sample_id.split("-")[:3]).upper()

In [ ]:
X_mRNA = reduced_shape_df
X_methylation = reduced_shape_df1
X_mutation = reduced_shape_df3
X_rppa = reduced_shape_df2
y = clinical_data['OS_STATUS'].map({'0:LIVING': 0, '1:DECEASED': 1})

# Convert omics data sample IDs
X_mRNA.index = [standardize_sample_id(s) for s in X_mRNA.index]
X_methylation.index = [standardize_sample_id(s) for s in X_methylation.index]
X_mutation.index = [standardize_sample_id(s) for s in X_mutation.index]
X_rppa.index = [standardize_sample_id(s) for s in X_rppa.index]

# Convert clinical data sample IDs (also force uppercase)
y.index = [standardize_sample_id(s).upper() for s in y.index]

In [ ]:
# Find common samples based on row indices (samples)
common_samples = X_mRNA.index.intersection(X_methylation.index).intersection(X_rppa.index).intersection(X_mutation.index)

# Display results
print("Number of common samples:", len(common_samples))
print("Sample IDs:", common_samples[:5])

In [ ]:
# Subset each dataset to include only common samples
df_mRNA = X_mRNA.loc[common_samples]
df_methylation = X_methylation.loc[common_samples]
df_rppa = X_rppa.loc[common_samples]
df_mutation = X_mutation.loc[common_samples]

In [ ]:
# Ensure index order is the same for all datasets
df_mRNA = df_mRNA.loc[common_samples].sort_index()
df_methylation = df_methylation.loc[common_samples].sort_index()
df_rppa = df_rppa.loc[common_samples].sort_index()
df_mutation = df_mutation.loc[common_samples].sort_index()
y = y.loc[common_samples].sort_index()

In [ ]:
df_mRNA = df_mRNA[~df_mRNA.index.duplicated(keep="first")]
df_methylation = df_methylation[~df_methylation.index.duplicated(keep="first")]
df_rppa = df_rppa[~df_rppa.index.duplicated(keep="first")]
df_mutation = df_mutation[~df_mutation.index.duplicated(keep="first")]
y = y[~y.index.duplicated(keep="first")]

df_methylation = df_methylation.reindex(common_samples)
df_rppa = df_rppa.reindex(common_samples)
df_mutation = df_mutation.reindex(common_samples)
y = y.reindex(common_samples)  # Ensure y is aligned too
df_mRNA = df_mRNA.reindex(common_samples)
print(df_mRNA.shape,df_rppa.shape,df_methylation.shape, df_mutation.shape,y.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score

# Define the MLP model
class SimpleMLP(nn.Module):
    def __init__(self, input_dim):  # Get input_dim as an argument
        super(SimpleMLP, self).__init__()
        self.fc1 = torch.nn.Linear(input_dim, 128).type(torch.float32) # Use input_dim
        self.fc2 = torch.nn.Linear(128, 1).type(torch.float32) # Output size should be 1 for binary classification
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)  # Apply ReLU activation
        x = self.fc2(x)
        x = self.sigmoid(x)  # Apply sigmoid for binary classification
        return x

# Train the model on each omics combination
def train_model(X_train, y_train, X_test, y_test):
    model = SimpleMLP(X_train.shape[1])  # Pass input dimension to model
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()

    # Convert to PyTorch tensors and ensure correct types
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)  # Reshape for BCELoss
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)  # Reshape for BCELoss

    # Training loop
    for epoch in range(100):  # You can adjust the number of epochs
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    # Prediction and AUC calculation
    with torch.no_grad():
        predictions = model(X_test)
        auc_score = roc_auc_score(y_test.numpy(), predictions.numpy())

    return auc_score

In [ ]:
import numpy as np
from itertools import combinations
from sklearn.model_selection import train_test_split


# Step 4: Define the omics sets with the filtered and aligned data
omics_sets = [
    ("mRNA", df_mRNA),
    ("Methylation", df_methylation),
    ("CNA", df_mutation),
    ("Proteomics", df_rppa)
]

# Step 5: Create all possible combinations
omics_combinations = []
for r in range(1, len(omics_sets) + 1):
    for combo in combinations(omics_sets, r):
        name = "+".join([x[0] for x in combo])

        # Get the intersection of samples for this combination
        common_indices = combo[0][1].index
        for i in range(1, len(combo)):
            common_indices = common_indices.intersection(combo[i][1].index)

        # Convert to a sorted list to ensure proper alignment
        common_indices = sorted(list(common_indices))

        # Subset and align each dataset using `.reindex()`
        subset_data = [x[1].reindex(common_indices) for x in combo]

        # Ensure all datasets have the exact same number of samples before concatenation
        row_counts = [df.shape[0] for df in subset_data]
        if len(set(row_counts)) != 1:
            print(f"Skipping {name} due to mismatch: {row_counts}")
            continue  # Skip if sample sizes don't match

        # Concatenate datasets along the feature axis
        data = np.concatenate(subset_data, axis=1)

        omics_combinations.append((name, data, common_indices))

# Step 6: Train and evaluate models
payoff_matrix = {}  # Store results
for name, data, common_indices in omics_combinations:
    y_filtered = y.reindex(common_indices)  # Ensure y is aligned

    if data.shape[0] != y_filtered.shape[0]:  # Final consistency check
        print(f"Skipping {name}: X has {data.shape[0]}, y has {y_filtered.shape[0]}")
        continue

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        data, y_filtered.values.ravel(), test_size=0.2, random_state=42
    )

    # Compute AUC for each combination
    auc_score = train_model(X_train, y_train, X_test, y_test)
    payoff_matrix[name] = auc_score

# Display results
print("Payoff Matrix (AUC scores):")
for key, value in payoff_matrix.items():
    print(f"{key}: {value:.3f}")

In [ ]:
import numpy as np

# Define the omics types
omics_types = ["mRNA", "Methylation", "CNA", "Proteomics"]

# Initialize a 3x3 matrix with NaNs (to identify missing values)
payoff_matrix_np = np.full((4, 4), np.nan)

# Fill the matrix with AUC scores from the dictionary
for i, omic_i in enumerate(omics_types):
    for j, omic_j in enumerate(omics_types):
        if i == j:
            # Self-payoff (individual AUC score)
            key = omic_i
        else:
            # Pairwise interactions
            key = f"{omic_i}+{omic_j}"
            reverse_key = f"{omic_j}+{omic_i}"  # Ensure symmetry

        # Assign AUC score if available
        if key in payoff_matrix:
            payoff_matrix_np[i, j] = payoff_matrix[key]
        elif reverse_key in payoff_matrix:
            payoff_matrix_np[i, j] = payoff_matrix[reverse_key]
        else:
            payoff_matrix_np[i, j] = 0  # Assign zero if no value exists

# Convert NaNs to zero (if needed)
payoff_matrix_np = np.nan_to_num(payoff_matrix_np)

# Print results
print("Omics Strategies:", omics_types)
print("Full Payoff Matrix (NumPy format):")
print(payoff_matrix_np)

In [ ]:
import numpy as np
import pandas as pd

# Définir les types d'omics
omics_types = ["mRNA", "Methylation", "CNA", "Proteomics"]

# Initialiser une matrice 4x4 avec des NaN
payoff_matrix_np = np.full((len(omics_types), len(omics_types)), np.nan)

# Remplir la matrice avec les scores AUC du dictionnaire
for i, omic_i in enumerate(omics_types):
    for j, omic_j in enumerate(omics_types):
        if i == j:
            # Score AUC individuel
            key = omic_i
        else:
            # Score AUC des interactions entre omics
            key = f"{omic_i}+{omic_j}"
            reverse_key = f"{omic_j}+{omic_i}"  # Assurer la symétrie

        # Assigner le score AUC si disponible, sinon 0
        payoff_matrix_np[i, j] = payoff_matrix.get(key, payoff_matrix.get(reverse_key, 0))

# Convertir les NaN en zéros (si nécessaire)
payoff_matrix_np = np.nan_to_num(payoff_matrix_np)

# Afficher la matrice sous forme de DataFrame pour plus de lisibilité
payoff_df = pd.DataFrame(payoff_matrix_np, index=omics_types, columns=omics_types)
print("Matrice complète des gains :")
print(payoff_df)

In [ ]:
# Initial strategy distribution (random start)
x = np.array([0.25, 0.25, 0.25, 0.25])

# Evolutionary process (Replicator Dynamics)
for _ in range(100):
    fitness = np.dot(payoff_matrix_np, x)  # Compute fitness
    avg_fitness = np.dot(x, fitness)  # Compute average fitness
    x = x * (fitness / avg_fitness)  # Update strategy distribution
    x = x / np.sum(x)  # Normalize

print("Optimal Omics Contribution:", x)